# 2: Exploratory Data Analysis

This notebook explores the merged dataset and turns it into a set of readable descriptive findings. 


## 2.1 Setup and load the merged dataset

This section loads the merged dataset and prepares the figure output folder so the rest of the notebook can focus on analysis rather than file setup.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.ticker import PercentFormatter

NOTEBOOK_DIR = Path.cwd().resolve()
# Support running the notebook either inside notebooks/ or from the project root.
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
# EDA always starts from the slim merged dataset created in notebook 01.
DATASET_PATH = ROOT / "data" / "processed" / "dataset_final.csv"
OUTPUTS_DIR = ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# A white grid theme keeps percentage plots readable without overwhelming the report figures.
sns.set_theme(style="whitegrid", context="talk")

print(f"Project root: {ROOT}")
print(f"Dataset path: {DATASET_PATH.relative_to(ROOT)}")
print(f"Figures directory: {FIGURES_DIR.relative_to(ROOT)}")


Project root: /Users/berenaydogan/Documents/4.2/DSA210/Project/DSA210-Project
Dataset path: data/processed/dataset_final.csv
Figures directory: outputs/figures


In [ ]:
# Parse the key timestamp columns up front so temporal grouping and plotting use real datetimes.
date_columns = ["service_date", "scheduled_arrival", "weather_time"]

assert DATASET_PATH.exists(), "Run 01_data_collection.ipynb first so data/processed/dataset_final.csv exists."

df = pd.read_csv(DATASET_PATH, parse_dates=date_columns)
dataset_row_count = len(df)
dataset_column_count = len(df.columns)

season_order = [
    "Winter (Jan window)",
    "Spring (Apr window)",
    "Summer (Jul window)",
    "Autumn (Oct window)",
]
# Preserve the intended seasonal sequence instead of letting alphabetical sorting scramble the windows.
df["season_window"] = pd.Categorical(
    df["season_window"],
    categories=season_order,
    ordered=True,
)

# These checks confirm the merge notebook produced a fully labeled, analysis table with weather matched to every row.
assert df["season_window"].notna().all(), "Every row should map to a seasonal window."
assert df["weather_matched"].all(), "The merged dataset should have full weather coverage."

print(f"Loaded rows: {dataset_row_count:,}")
print(f"Loaded columns: {dataset_column_count:,}")
display(df.head())


Loaded rows: 90,818
Loaded columns: 21


,service_date,season_window,scheduled_arrival,scheduled_arrival_date,delay_minutes,delayed,scheduled_arrival_hour,scheduled_arrival_weekday,scheduled_arrival_weekday_name,scheduled_arrival_month,...,stop_name,train_type,line_name,operator,weather_time,temperature_2m,precipitation,snowfall,wind_speed_10m,weather_matched
0,2025-01-01,Winter (Jan window),2025-01-01 01:25:00,2025-01-01,-0.200000,0,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True
1,2025-01-01,Winter (Jan window),2025-01-01 01:27:00,2025-01-01,1.000000,1,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True
2,2025-01-01,Winter (Jan window),2025-01-01 01:34:00,2025-01-01,1.300000,1,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True
3,2025-01-01,Winter (Jan window),2025-01-01 01:35:00,2025-01-01,-0.650000,0,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True
4,2025-01-01,Winter (Jan window),2025-01-01 01:52:00,2025-01-01,0.183333,1,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True


## 2.2 Dataset overview and integrity checks

This section checks the basic shape of the dataset, confirms the key timestamps line up as expected, and looks for missing values before I start interpreting plots.


In [ ]:
scheduled_date = df["scheduled_arrival"].dt.strftime("%Y-%m-%d")
service_date = df["service_date"].dt.strftime("%Y-%m-%d")
# Count rows where scheduled arrival spills past midnight so the report can explain any mismatch in the service date.
overnight_spillover_rows = (scheduled_date != service_date).sum()

overview_df = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "unique train types",
            "stations in the Lausanne area",
            "unique lines",
            "unique operators",
            "service date start",
            "service date end",
            "scheduled-arrival start",
            "scheduled-arrival end",
            "weather coverage",
            "overnight spillover rows",
        ],
        "value": [
            dataset_row_count,
            dataset_column_count,
            df["train_type"].nunique(),
            df["stop_name"].nunique(),
            df["line_name"].nunique(),
            df["operator"].nunique(),
            df["service_date"].min().date().isoformat(),
            df["service_date"].max().date().isoformat(),
            df["scheduled_arrival"].min().isoformat(sep=" "),
            df["scheduled_arrival"].max().isoformat(sep=" "),
            f"{df['weather_matched'].mean():.2%}",
            int(overnight_spillover_rows),
        ],
    }
)

station_counts = (
    df["stop_name"]
    .value_counts()
    .rename_axis("stop_name")
    .reset_index(name="rows")
)

display(overview_df)
display(station_counts)


,metric,value
0,rows,90818
1,columns,21
2,unique train types,9
3,stations in the Lausanne area,4
4,unique lines,27
5,unique operators,2
6,service date start,2025-01-01
7,service date end,2025-10-30
8,scheduled-arrival start,2025-01-01 01:25:00
9,scheduled-arrival end,2025-10-31 01:52:00


,stop_name,rows
0,Lausanne,63942
1,Romanel-sur-Lausanne,10749
2,Lausanne-Chauderon,10749
3,Lausanne-Flon,5378


In [ ]:
missing_summary = (
    df.isna()
    .sum()
    .rename("missing_rows")
    .to_frame()
    # Use assign so the percentage is computed from the grouped table without creating a separate temporary object.
    .assign(missing_pct=lambda x: x["missing_rows"] / len(df))
)
missing_summary = missing_summary[missing_summary["missing_rows"] > 0].sort_values(
    "missing_rows", ascending=False
)

day_gap = (
    df["scheduled_arrival"].dt.normalize() - df["service_date"].dt.normalize()
).dt.days

timestamp_checks = pd.DataFrame(
    {
        "check": [
            "Rows missing critical timestamps",
            "Rows missing weather timestamp",
            "Rows missing any hourly weather variable",
            "Rows with weather_matched = False",
            "Rows with scheduled arrival on service date",
            "Rows with scheduled arrival one day after service date",
            "Rows with any other service date gap",
        ],
        "value": [
            int(df[["service_date", "scheduled_arrival", "delay_minutes", "delayed"]].isna().any(axis=1).sum()),
            int(df["weather_time"].isna().sum()),
            # Run this check across the four weather columns to confirm the merge did not leave partial hourly data behind.
            int(df[["temperature_2m", "precipitation", "snowfall", "wind_speed_10m"]].isna().any(axis=1).sum()),
            int((~df["weather_matched"]).sum()),
            int((day_gap == 0).sum()),
            int((day_gap == 1).sum()),
            int((~day_gap.isin([0, 1])).sum()),
        ],
    }
)

if missing_summary.empty:
    print(
        "The dataset has no missing values in the retained analysis columns. "
    )
else:
    print(
        "The remaining missing values are limited to retained analysis fields that still need interpretation. "
    )
display(missing_summary)
display(timestamp_checks)


The slim analysis dataset has no missing values in the retained analysis columns. The overnight spillover rows reflect trains scheduled shortly after midnight, not corrupted timestamps.


,missing_rows,missing_pct


,check,value
0,Rows missing critical timestamps,0
1,Rows missing weather timestamp,0
2,Rows missing any hourly weather variable,0
3,Rows with weather_matched = False,0
4,Rows with scheduled arrival on service date,87620
5,Rows with scheduled arrival one day after serv...,3198
6,Rows with any other service date gap,0
